# Co-occurrence RRF Blend — Reference Example

This notebook demonstrates blending the two-tower model with co-occurrence retrieval via reciprocal rank fusion (RRF). It is a **reference example** — production serving uses model-only (Feature Views → Vector Search).

**What this notebook does:**
* Builds a co-occurrence matrix from training sessions (item pairs viewed together)
* Loads the trained two-tower model from MLflow
* Evaluates three approaches on has-history novel rows:
  1. Model alone (user tower + item embeddings)
  2. Co-occurrence alone (session-level item overlap)
  3. RRF blend: `score = 1/(k + rank_model) + w × 1/(k + rank_cooc)`
* Sweeps blend weight w on eval (Nov 1–3), validates on test (Nov 4–6)

**Serving implication:** The blend requires a co-occurrence neighbor lookup store (e.g., Lakebase) alongside Vector Search. The model-only path needs no such dependency.

Requires GPU compute. Run `03_two_tower_final` first to produce the trained model and Parquet exports.

In [0]:
%pip install databricks-feature-engineering>=0.16.0 torch scipy --quiet
dbutils.library.restartPython()

In [0]:
import os, sys, yaml, json, math, time as _time, random, glob, pickle, importlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as TF
import pyarrow.parquet as pq
from scipy.sparse import csr_matrix
from pyspark.sql import functions as F

config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

CATALOG = config["catalog"]
SCHEMA = config["schema"]
EMBED_DIM = config["embed_dim"]
BATCH_SIZE = config["batch_size"]

VOLUME_BASE = config.get("volume_base", f"/Volumes/{config.get('volume_catalog', 'shared')}/{config.get('volume_schema', 'mlc_schema')}/{config.get('volume', 'recommender_data')}")
TRAIN_PARQUET_PATH = f"{VOLUME_BASE}/session_train"
EVAL_PARQUET_PATH = f"{VOLUME_BASE}/session_eval"
TEST_PARQUET_PATH = f"{VOLUME_BASE}/session_test"
CLICKSTREAM_TABLE = f"{CATALOG}.{SCHEMA}.ecommerce_clickstream_full"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Column lists (must match 04_two_tower_final) ----
MAX_RECENT_PRODUCTS = 10
MAX_RECENT_BRANDS = 5
ITEM_COLS = [
    "item_idx", "category_idx", "category_l2_idx", "category_l3_idx", "brand_idx",
    "item_log_price", "price_pct_rank", "price_pct_rank_l2",
    "item_popularity", "item_conversion_rate", "item_cart_to_purchase_rate",
    "item_recency_val",
]
USER_NUMERIC_COLS = [
    "session_event_count", "session_avg_price", "session_price_stddev",
    "session_total_value", "session_cart_value",
    "session_view_count", "session_cart_count", "session_purchase_count",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]
SEQ_COLS = [
    "recent_product_ids", "recent_product_mask",
    "recent_brand_ids", "recent_brand_mask",
]
ITEM_COL_MAP = {
    "item_idx": "item_id", "category_idx": "category_id",
    "category_l2_idx": "category_l2_id", "category_l3_idx": "category_l3_id",
    "brand_idx": "brand_id", "item_log_price": "log_price",
    "price_pct_rank": "price_pct_rank", "price_pct_rank_l2": "price_pct_rank_l2",
    "item_popularity": "item_popularity", "item_conversion_rate": "item_conversion_rate",
    "item_cart_to_purchase_rate": "item_cart_to_purchase_rate",
    "item_recency_val": "item_recency",
}
SCALAR_COLS = list(ITEM_COL_MAP.keys()) + USER_NUMERIC_COLS

print(f"Device: {device}")
print(f"Volume: {VOLUME_BASE}")

In [0]:
# ---------------------------------------------------------------------------
# Build co-occurrence matrix from training sessions.
# For each session, every pair of viewed items contributes +1.
# Stored as a symmetric sparse CSR matrix (vocab_idx × vocab_idx).
# Saved to volume for reuse.
# ---------------------------------------------------------------------------
import scipy.sparse as sp

def build_cooccurrence_matrix():
    """Build item co-occurrence matrix from training Parquet."""
    # Load training data — need session_id + product_ids per session
    train_pdf = pd.read_parquet(TRAIN_PARQUET_PATH, columns=["item_idx"])
    # We need session grouping — read from clickstream table instead
    # to get raw session → item mappings
    print("Loading training sessions from clickstream...")
    sdf = spark.table(CLICKSTREAM_TABLE) \
        .filter("event_type = 'view'") \
        .filter("CAST(event_time AS DATE) BETWEEN '2019-10-14' AND '2019-10-31'") \
        .select("user_session", "product_id")

    # Build vocab from training Parquet (item_idx mapping)
    item_feat = pd.read_parquet(f"{VOLUME_BASE}/item_features")
    # product_id → item_idx mapping
    # We need the reverse: product_id → item_idx
    # Load from the original training table
    _prod = spark.table(f"{CATALOG}.{SCHEMA}.{config['product_train_table']}")
    pid_idx = _prod.select("product_id").distinct().toPandas()["product_id"].values
    # The vocab is built sorted, so item_idx = rank(product_id) + 1
    pid_sorted = sorted(pid_idx)
    pid_to_idx = {pid: i + 1 for i, pid in enumerate(pid_sorted)}
    del pid_sorted

    max_idx = int(train_counts['item_idx'].max())
    print(f"Vocab size: {max_idx + 1}")

    # Collect session → item_idx lists
    print("Collecting session-item pairs...")
    session_items = sdf.rdd \
        .map(lambda r: (r['user_session'], pid_to_idx.get(r['product_id'], 0))) \
        .filter(lambda x: x[1] > 0) \
        .groupByKey() \
        .mapValues(list) \
        .collect()

    print(f"Sessions: {len(session_items):,}")

    # Build co-occurrence matrix
    print("Building co-occurrence matrix...")
    rows, cols, vals = [], [], []
    for sess, items in session_items:
        unique = sorted(set(items))
        if len(unique) < 2:
            continue
        for i in range(len(unique)):
            for j in range(i + 1, len(unique)):
                rows.extend([unique[i], unique[j]])
                cols.extend([unique[j], unique[i]])
                vals.extend([1, 1])

    cooc = sp.csr_matrix(
        (np.array(vals, dtype=np.float32), (np.array(rows), np.array(cols))),
        shape=(max_idx + 1, max_idx + 1)
    )
    cooc.eliminate_zeros()
    print(f"Co-occurrence matrix: {cooc.shape}, nnz={cooc.nnz:,}")

    # Save
    with open(f"{VOLUME_BASE}/cooc_matrix.pkl", "wb") as f:
        pickle.dump(cooc, f)
    print(f"Saved to {VOLUME_BASE}/cooc_matrix.pkl")
    return cooc

# Build or load
_coc_path = f"{VOLUME_BASE}/cooc_matrix.pkl"
if os.path.exists(_coc_path):
    print("Loading existing co-occurrence matrix...")
    with open(_coc_path, "rb") as f:
        cooc_matrix = pickle.load(f)
    print(f"Co-occurrence matrix: {cooc_matrix.shape}, nnz={cooc_matrix.nnz:,}")
else:
    cooc_matrix = build_cooccurrence_matrix()

In [0]:
# ---------------------------------------------------------------------------
# Load the trained two-tower model from 04_two_tower_final.
# If the model is in kernel state (run in same session), skip loading.
# Otherwise, rebuild architecture and load traced towers from MLflow.
# ---------------------------------------------------------------------------

def masked_mean_pool(embeddings, mask):
    mask_expanded = mask.unsqueeze(-1)
    summed = (embeddings * mask_expanded).sum(dim=1)
    counts = mask.sum(dim=1, keepdim=True).clamp(min=1)
    return summed / counts

class SharedEmbeddings(nn.Module):
    def __init__(self, item_vocab_size, brand_vocab_size, embed_dim):
        super().__init__()
        self.item_emb = nn.Embedding(item_vocab_size, embed_dim, padding_idx=0)
        self.brand_emb = nn.Embedding(brand_vocab_size, embed_dim // 2, padding_idx=0)

class ItemTower(nn.Module):
    def __init__(self, shared_emb, category_vocab_size, category_l2_vocab_size,
                 category_l3_vocab_size, embed_dim=64):
        super().__init__()
        self.shared = shared_emb
        self.category_emb = nn.Embedding(category_vocab_size, embed_dim // 2)
        self.category_l2_emb = nn.Embedding(category_l2_vocab_size, embed_dim // 4)
        self.category_l3_emb = nn.Embedding(category_l3_vocab_size, embed_dim // 4)
        numeric_dim = 7
        mlp_in = embed_dim + (embed_dim // 2) * 2 + (embed_dim // 4) * 2 + numeric_dim
        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, embed_dim),
        )
    def forward(self, item_id, category_id, category_l2_id, category_l3_id, brand_id,
                log_price, price_pct_rank, price_pct_rank_l2, item_popularity,
                item_conversion_rate, item_cart_to_purchase_rate, item_recency):
        x = torch.cat([
            self.shared.item_emb(item_id),
            self.category_emb(category_id),
            self.category_l2_emb(category_l2_id),
            self.category_l3_emb(category_l3_id),
            self.shared.brand_emb(brand_id),
            log_price.unsqueeze(-1), price_pct_rank.unsqueeze(-1),
            price_pct_rank_l2.unsqueeze(-1), item_popularity.unsqueeze(-1),
            item_conversion_rate.unsqueeze(-1), item_cart_to_purchase_rate.unsqueeze(-1),
            item_recency.unsqueeze(-1),
        ], dim=-1)
        return TF.normalize(self.mlp(x), p=2, dim=-1)

class UserTower(nn.Module):
    def __init__(self, shared_emb, embed_dim=64):
        super().__init__()
        self.shared = shared_emb
        numeric_dim = 12
        seq_dim = embed_dim + embed_dim // 2
        self.mlp = nn.Sequential(
            nn.Linear(numeric_dim + seq_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, embed_dim),
        )
    def forward(self, numeric, recent_product_ids, recent_product_mask,
                recent_brand_ids, recent_brand_mask):
        prod_emb = self.shared.item_emb(recent_product_ids)
        prod_pooled = masked_mean_pool(prod_emb, recent_product_mask)
        brand_emb = self.shared.brand_emb(recent_brand_ids)
        brand_pooled = masked_mean_pool(brand_emb, recent_brand_mask)
        x = torch.cat([numeric, prod_pooled, brand_pooled], dim=-1)
        return TF.normalize(self.mlp(x), p=2, dim=-1)

class TwoTowerModel(nn.Module):
    def __init__(self, shared_emb, item_tower, user_tower, temperature=0.05):
        super().__init__()
        self.shared_emb = shared_emb
        self.item_tower = item_tower
        self.user_tower = user_tower
        self.temperature = temperature

# ---- Build vocabs from training data (same as 04) ----
_prod = spark.table(f"{CATALOG}.{SCHEMA}.{config['product_train_table']}")
_sess = spark.table(f"{CATALOG}.{SCHEMA}.{config['session_train_table']}")
_common = sorted(set(_prod.columns) & set(_sess.columns))
_sess_only = [c for c in _sess.columns if c not in set(_prod.columns)]
train_df = _prod.join(_sess.select(*_common, *_sess_only), on=_common, how="inner")

def build_vocab(sdf, col):
    vals = [row[col] for row in sdf.select(col).distinct().collect() if row[col] is not None]
    return {v: i + 1 for i, v in enumerate(sorted(vals))}

item_vocab = build_vocab(train_df, "product_id")
category_vocab = build_vocab(train_df, "item_category_l1")
category_l2_vocab = build_vocab(train_df, "item_category_l2")
category_l3_vocab = build_vocab(train_df, "item_category_l3")
brand_vocab = build_vocab(train_df, "item_brand")

# ---- Instantiate model ----
shared = SharedEmbeddings(
    item_vocab_size=len(item_vocab) + 1,
    brand_vocab_size=len(brand_vocab) + 1,
    embed_dim=EMBED_DIM,
)
item_tower = ItemTower(
    shared_emb=shared,
    category_vocab_size=len(category_vocab) + 1,
    category_l2_vocab_size=len(category_l2_vocab) + 1,
    category_l3_vocab_size=len(category_l3_vocab) + 1,
    embed_dim=EMBED_DIM,
)
user_tower = UserTower(shared_emb=shared, embed_dim=EMBED_DIM)
model = TwoTowerModel(shared, item_tower, user_tower).to(device)

# ---- Try loading latest MLflow run's traced towers ----
import mlflow
experiment = mlflow.get_experiment_by_name(config["mlflow_experiment"])
if experiment:
    runs = mlflow.search_runs([experiment.experiment_id], order_by=["attributes.start_time DESC"], max_results=1)
    if len(runs) > 0:
        run_id = runs.iloc[0]["run_id"]
        print(f"Loading model from MLflow run: {run_id}")
        # Load traced towers
        mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="traced_towers/user_tower_traced.pt", dst_path="/tmp")
        mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="traced_towers/item_tower_traced.pt", dst_path="/tmp")
        # Load state from traced models
        traced_user = torch.jit.load("/tmp/user_tower_traced.pt", map_location=device)
        traced_item = torch.jit.load("/tmp/item_tower_traced.pt", map_location=device)
        model.user_tower.load_state_dict(traced_user.state_dict())
        model.item_tower.load_state_dict(traced_item.state_dict())
        print("Model loaded from MLflow traced towers")
    else:
        print("WARNING: No MLflow runs found. Ensure 04_two_tower_final has been run.")
else:
    print("WARNING: MLflow experiment not found. Ensure 04_two_tower_final has been run.")

model.eval()
print(f"Model ready on {device}")

In [0]:
# ---------------------------------------------------------------------------
# RRF blend evaluation: model + co-occurrence.
# rrf_score = 1/(k + rank_model) + w * 1/(k + rank_cooc)
# ---------------------------------------------------------------------------
K_RRF = 60
TOP_K = 150

def parquet_batch_to_model_inputs(batch, device):
    item_batch = {}
    for col, kwarg in ITEM_COL_MAP.items():
        dtype = torch.long if col.endswith("idx") else torch.float32
        item_batch[kwarg] = batch[col].to(dtype=dtype, device=device)
    user_numeric = torch.stack(
        [batch[c].float() for c in USER_NUMERIC_COLS], dim=-1
    ).to(device)
    user_batch = {
        "numeric": user_numeric,
        "recent_product_ids": batch["recent_product_ids"].long().to(device),
        "recent_product_mask": batch["recent_product_mask"].float().to(device),
        "recent_brand_ids": batch["recent_brand_ids"].long().to(device),
        "recent_brand_mask": batch["recent_brand_mask"].float().to(device),
    }
    return user_batch, item_batch

def build_item_embeddings(model, volume_base):
    item_features = pd.read_parquet(f"{volume_base}/item_features")
    _long = ['item_idx','category_idx','category_l2_idx','category_l3_idx','brand_idx']
    _float = ['item_log_price','price_pct_rank','price_pct_rank_l2',
              'item_popularity','item_conversion_rate','item_cart_to_purchase_rate','item_recency_val']
    model.eval()
    with torch.no_grad():
        args = [torch.tensor(item_features[c].values, dtype=torch.long, device=device) for c in _long]
        args += [torch.tensor(item_features[c].values, dtype=torch.float32, device=device) for c in _float]
        item_embs = TF.normalize(model.item_tower(*args), dim=1)
    ia = item_features['item_idx'].values
    mx = int(ia.max())
    i2p = torch.full((mx+1,), 0, dtype=torch.long, device=device)
    i2p[torch.tensor(ia, device=device)] = torch.arange(len(ia), device=device)
    return item_embs, i2p, ia, mx

# ---- Build item embeddings ----
item_embs, i2p, ia, mx = build_item_embeddings(model, VOLUME_BASE)
N_items = len(ia)
print(f"Item embeddings: {N_items:,} items")

# ---- Build co-occurrence top-K neighbor index on GPU ----
_cooc_sz = cooc_matrix.shape[0]
print(f"Building top-{TOP_K} neighbor index from {cooc_matrix.nnz:,} nnz...")
t0 = _time.time()
_nb = np.zeros((_cooc_sz, TOP_K), dtype=np.int64)
_sc = np.zeros((_cooc_sz, TOP_K), dtype=np.float32)
for i in range(_cooc_sz):
    s, e = cooc_matrix.indptr[i], cooc_matrix.indptr[i + 1]
    n = e - s
    if n == 0: continue
    if n <= TOP_K:
        _nb[i, :n] = cooc_matrix.indices[s:e]
        _sc[i, :n] = cooc_matrix.data[s:e]
    else:
        idx = np.argpartition(-cooc_matrix.data[s:e], TOP_K)[:TOP_K]
        _nb[i] = cooc_matrix.indices[s:e][idx]
        _sc[i] = cooc_matrix.data[s:e][idx]
cooc_nb = torch.tensor(_nb, dtype=torch.long, device=device)
cooc_sc = torch.tensor(_sc, dtype=torch.float32, device=device)
print(f"Neighbor index built in {_time.time()-t0:.1f}s")


def rrf_blend_eval(model, parquet_path, w=1.0, label="eval"):
    """RRF blend of model + co-occurrence on has-history novel rows."""
    item_embs, i2p, ia, mx = build_item_embeddings(model, VOLUME_BASE)
    mrr_blend, mrr_model, mrr_cooc = 0.0, 0.0, 0.0
    r10_blend, cnt = 0, 0

    for fpath in sorted(glob.glob(f"{parquet_path}/*.parquet")):
        for rb in pq.ParquetFile(fpath).iter_batches(2048,
                columns=list(set(SCALAR_COLS + SEQ_COLS))):
            iidx = rb.column('item_idx').to_numpy(zero_copy_only=False)
            rp = np.stack(rb.column('recent_product_ids').to_pandas().values)
            rm = np.stack(rb.column('recent_product_mask').to_pandas().values)
            B = len(iidx)
            has_hist = rm.sum(axis=1) > 0
            is_rep = np.zeros(B, dtype=bool)
            for j in range(rp.shape[1]):
                is_rep |= ((rp[:,j]==iidx) & (rm[:,j]>0))
            nh = np.where((~is_rep) & has_hist)[0]
            if len(nh) == 0: continue

            batch = {}
            for c in SCALAR_COLS:
                batch[c] = torch.from_numpy(rb.column(c).to_numpy(zero_copy_only=False))
            for c in SEQ_COLS:
                batch[c] = torch.from_numpy(np.stack(rb.column(c).to_pandas().values))
            ub, _ = parquet_batch_to_model_inputs(batch, device)

            with torch.no_grad():
                ue = model.user_tower(**ub)
            ue = ue[nh]
            Bh = len(nh)
            _ti = torch.tensor(iidx[nh], dtype=torch.long, device=device)
            _ti_pos = i2p[_ti.clamp(0,mx)].clamp(min=0)
            _rp = torch.tensor(rp[nh], dtype=torch.long, device=device)
            _rm = torch.tensor(rm[nh], dtype=torch.float32, device=device)

            # Model scores
            sc_m = ue @ item_embs.T
            for j in range(_rp.shape[1]):
                v = _rm[:,j] > 0; r = torch.where(v)[0]
                if len(r) > 0:
                    sc_m[r, i2p[_rp[r,j].clamp(0,mx)].clamp(min=0)] = -float('inf')

            # Co-occ scores via scatter-add
            sc_c = torch.zeros(Bh, N_items, device=device)
            for j in range(_rp.shape[1]):
                v = _rm[:,j] > 0; r = torch.where(v)[0]
                if len(r) == 0: continue
                hist_idx = _rp[r,j].clamp(0, _cooc_sz-1)
                nb_ids = cooc_nb[hist_idx]
                nb_scores = cooc_sc[hist_idx]
                nb_pos = i2p[nb_ids.clamp(0,mx)].clamp(min=0)
                sc_c[r.unsqueeze(1).expand_as(nb_pos).reshape(-1),
                     nb_pos.reshape(-1)] += nb_scores.reshape(-1)
            for j in range(_rp.shape[1]):
                v = _rm[:,j] > 0; r = torch.where(v)[0]
                if len(r) > 0:
                    sc_c[r, i2p[_rp[r,j].clamp(0,mx)].clamp(min=0)] = -float('inf')

            # RRF
            rank_m = sc_m.argsort(dim=1, descending=True).argsort(dim=1) + 1
            rank_c = sc_c.argsort(dim=1, descending=True).argsort(dim=1) + 1
            rrf = 1.0/(K_RRF + rank_m.float()) + w/(K_RRF + rank_c.float())

            for scores, acc_name in [(rrf, 'blend'), (sc_m, 'model'), (sc_c, 'cooc')]:
                tgt = scores[torch.arange(Bh, device=device), _ti_pos]
                ranks = (scores > tgt.unsqueeze(1)).sum(1) + 1
                if acc_name == 'blend':
                    mrr_blend += (1.0/ranks.float()).sum().item()
                    r10_blend += int((ranks <= 10).sum().item())
                elif acc_name == 'model':
                    mrr_model += (1.0/ranks.float()).sum().item()
                else:
                    mrr_cooc += (1.0/ranks.float()).sum().item()
            cnt += Bh

    print(f"\n{'='*70}")
    print(f"  RRF BLEND — {label.upper()} (w={w}, k={K_RRF})")
    print(f"  {cnt:,} has-history novel rows")
    print(f"{'='*70}")
    print(f"  Blend:  MRR={mrr_blend/cnt:.4f}  R@10={r10_blend/cnt:.4f}")
    print(f"  Model:  MRR={mrr_model/cnt:.4f}")
    print(f"  Co-occ: MRR={mrr_cooc/cnt:.4f}")
    return {'blend_mrr': mrr_blend/cnt, 'model_mrr': mrr_model/cnt, 'cooc_mrr': mrr_cooc/cnt}

print("RRF blend evaluation function defined")

In [0]:
# ---------------------------------------------------------------------------
# Sweep blend weight w on eval (Nov 1–3), then validate on test (Nov 4–6).
# ---------------------------------------------------------------------------

# ---- Sweep w on eval ----
print("="*70)
print("  SWEEP: blend weight w on Nov 1–3 eval")
print("="*70)
eval_blend_results = {}
for w in [0.0, 0.25, 0.50, 0.75, 1.0, 1.5, 2.0, 3.0]:
    r = rrf_blend_eval(model, EVAL_PARQUET_PATH, w=w, label=f"Eval w={w}")
    eval_blend_results[w] = r
    print(f"  w={w:.2f}: Blend={r['blend_mrr']:.4f}  Model={r['model_mrr']:.4f}  Co-occ={r['cooc_mrr']:.4f}")

# Pick best w
best_w = max(eval_blend_results, key=lambda w: eval_blend_results[w]['blend_mrr'])
print(f"\nBest w = {best_w} (MRR = {eval_blend_results[best_w]['blend_mrr']:.4f})")

# ---- Validate on held-out test ----
print(f"\n{'='*70}")
print(f"  HELD-OUT TEST: Nov 4–6 (w={best_w})")
print(f"{'='*70}")
test_blend = rrf_blend_eval(model, TEST_PARQUET_PATH, w=best_w, label="Nov 4–6 Test")

# ---- Summary table ----
print(f"\n{'='*70}")
print(f"  FINAL SUMMARY: EVAL vs TEST")
print(f"{'='*70}")
print(f"  {'Method':20s} {'Eval MRR':>10s}  {'Test MRR':>10s}  {'Δ':>8s}")
for name, e, t in [
    ("Model alone", eval_blend_results[best_w]['model_mrr'], test_blend['model_mrr']),
    ("Co-occ alone", eval_blend_results[best_w]['cooc_mrr'], test_blend['cooc_mrr']),
    (f"Blend w={best_w}", eval_blend_results[best_w]['blend_mrr'], test_blend['blend_mrr']),
]:
    print(f"  {name:20s} {e:10.4f}  {t:10.4f}  {t-e:+8.4f}")
print(f"{'='*70}")